Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public
License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later
version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied
warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see
<https://www.gnu.org/licenses/>.

# The Steering Term of gr-fec's Turning Theory

STATE.md item 8 (e), under the maintainer's ruling "Pause + equations"
([gr-fec.md § 12.4](../docs/research/gr-fec.md#124-recommendations-for-the-maintainer-to-decide), recommendation 2):
extend the turning theory of 2026-10-26 ([journal](../docs/journal/2026-10-26.md#gr-fecs-worm-by-equation-its-turning-rule-predicted-before-it-runs),
[#112](https://github.com/ai-quokka-wannabe/iso-lab/pull/112)) with a *steering term* that predicts a worm's drift
from three things: the share of its steering decisions that point home, the noise of its lateral sense, and the size
of its turn. The derivation is on paper here, the predictions are tabulated before any simulation, and the test runs
gr-fec's own code, unedited, under an evaluation-side wrapper that imposes the share. Pre-registered in the journal of
2026-11-08, § gr-fec's Worm by Equation: The Steering Term, Predicted Before It Runs. A toy world: nothing here is
evidence about *C. elegans*. This notebook explores; nothing the benchmark depends on lives here.

**The question.** Given the share p of decisions whose map's choice points towards the warm side (home), a lateral
sense with Gaussian noise of standard deviation σ, and a turn of k radians per step, how fast does an undriven
e45-family worm drift up the thermal gradient of gr-fec's plate?

## 1. The mechanism, read from the sandbox's code

Every fact below is read from `reference/gr-fec/src/` at the pinned archive; the file and the line's content are named
so that a reader can check them.

- **The body** (`embodied.py`, `Body`): each step is one second; the worm moves `SPEED` = 0.2 mm along its heading;
  the heading gets Gaussian noise of standard deviation `HEAD_SD` = 0.05 rad every step; a pirouette redraws the
  heading uniformly on [0, 2π) before that noise is added (`act`: `np.where(pirouette, uniform, head) + HEAD_SD * N`).
- **The lateral sense** (`steer.py`, `DirBody.sense`): the temperature one millimetre to the worm's left minus one
  millimetre to its right, plus Gaussian noise of standard deviation 0.02. On the thermal plate (`World(thermal=True)`,
  `embodied.py`: `t = 20 + 5 x / 25`) the temperature rises by G_T = 0.2 °C per mm towards +x, so the noise-free
  reading at heading θ (measured from +x) is m(θ) = G_T · [(x − sin θ) − (x + sin θ)] = −2 G_T sin θ = −0.4 sin θ:
  the left side is warmer when the worm heads into −y, which is when a left turn brings it towards +x.
- **The turn** (`steer.py`, `DirBody.act`): `head += 0.15 * steer` every step, with `steer` ∈ {−1, 0, +1} held from
  one decision to the next; so one decision turns the worm by up to 10 × 0.15 = 1.5 rad. The lab's H14 series scaled
  that 0.15 ([H14](../docs/hypotheses/README.md#h14-graded-slow-knobs-change-detecting-senses-and-individuality)), which is why
  the turn's size is a parameter here.
- **The decision** (`steer.py`, `SteeringBrain.step`, mode `greedy`): every 10 steps (`tick % 10 == 0`). The lateral
  reading is thresholded at ±0.01: above, the left is warmer; below −0.01, the right; between, undecided and
  `steer = 0`. The map chooses "warmer" or "colder" (whichever neighbouring concept is worth more by at least 0.02);
  wanting warmer with the left warmer, or colder with the right warmer, gives `steer = +1` (a left turn), the two
  other decided cases give −1.
- **The pirouette layer** (`evolve.py`, `EvoBrain.step`): P = 0.3 / (1 + exp((dD/s0 + 1)/τ)). With every drive gene
  at zero the drive D is identically zero (the learned weights start at zero and the reward is zero, so they stay
  there; `g_gut` = 0 removes the gut term), dD = 0, and the pirouette probability per step is the constant
  λ₀ = 0.3 / (1 + e^{1/τ}); at τ = 0.3, λ₀ = 0.01033 per second, the rate the 2026-10-26 run measured at its weakest
  gradients (0.0102 to 0.0105).

## 2. The derivation

**Assumptions**, each stated so that it can be blamed.

- A1. The map's choice at each decision is an independent coin: "warmer" with probability p, "colder" with 1 − p,
  independent of the heading, of earlier choices and of the lateral reading. In a learned map the choice depends on the
  worm's position through its temperature concept; here the share is imposed, which is the point of step (e).
- A2. The lateral noise is Gaussian, independent between decisions and of the heading noise. In the sandbox it is drawn
  fresh each step, so this holds exactly.
- A3. The plate has no wall and the gradient is uniform, so the heading is the only state that matters for the drift:
  m(θ) does not depend on position. The wall is removed by setting the body's radius to 10¹² mm, as on 2026-10-26.
- A4. The drive is zero, so pirouettes are a Poisson-like coin at λ₀ per step, independent of the heading. The
  steering term is derived for this baseline; composing it with a driven turning rule (where λ depends on the heading
  through the odour gradient) is left for a later step and is said so.
- A5. The heading is discretised on 720 angles, the heading noise is applied as a circular convolution and the turn as
  a Fourier-space rotation. After one convolution with a kernel of standard deviation 0.05 rad (5.7 grid cells) the
  density is band-limited to the grid's resolution, so these are exact to rounding, not approximations in the sense the
  2026-10-26 theory's noise averaging was.

**The steering probabilities** (closed form). Let Φ be the standard normal distribution function and a = 0.01 the
threshold. At heading θ, with m = m(θ) = −2 G_T sin θ,

- P(left warmer) = Φ((m − a)/σ), P(right warmer) = Φ((−m − a)/σ), P(undecided) = 1 − the two;
- P(steer = +1 | θ) = p Φ((m − a)/σ) + (1 − p) Φ((−m − a)/σ);
- P(steer = −1 | θ) = p Φ((−m − a)/σ) + (1 − p) Φ((m − a)/σ).

**The steering term** is the expected rotation per step that follows, k times the difference:

> E[Δθ_steer | θ] = k (2p − 1) [Φ((m(θ) − a)/σ) − Φ((−m(θ) − a)/σ)].

It is an odd function of sin θ; for p > ½ it rotates the heading towards +x (for θ in (0, π), m < 0, the bracket is
negative, the rotation clockwise), for p < ½ away, and for p = ½ it vanishes identically. The three inputs enter
through three factors: the share through (2p − 1), the noise through the bracket's steepness and its dead zone, and
the turn's size as the prefactor.

**The chain.** The term alone does not give the drift, because the turn is held for ten steps while pirouettes and
heading noise act, and because the drift is v⟨cos θ⟩ over the stationary heading, not a rotation rate. Let g(θ) be the
heading density at a decision step. The decision splits it into three densities f_s(θ) = g(θ) P(s | θ), s ∈ {−1, 0, +1}.
Each of the ten following steps applies to each f_s, in the sandbox's order,

1. the turn: f_s(θ) ← f_s(θ − k s), a rotation;
2. the pirouette: f_s ← (1 − λ₀) f_s + λ₀ ∫f_s / 2π, the mass redrawn uniformly keeps its steer;
3. the heading noise: f_s ← f_s ∗ N(0, 0.05²), a circular convolution, under which a uniform density is unchanged;

and the step's movement contributes v Σ_s ∫ f_s(θ) cos θ dθ to the drift. After ten steps the three sum to the next
decision's g. Iterated to its fixed point (a contraction, since each cycle mixes a fraction ≥ 1 − (1 − λ₀)^{10} of the
mass into the uniform density), the drift is

> drift = (v / 10) Σ_{steps} Σ_s ∫ f_s(θ) cos θ dθ, in mm per second.

**Consequences that need no computation**, each a check on the code below and a prediction about the sandbox:

- C1. drift(p = ½) = 0 exactly: with s = ±1 equiprobable at every heading, the reflection θ → π − θ maps the chain onto
  itself (it fixes sin θ, hence every P(s | θ), and swaps the two rotations), so the stationary density is symmetric
  under it and ⟨cos θ⟩ = 0.
- C2. drift(1 − p, σ, k) = −drift(p, σ, k): the same reflection composed with p → 1 − p.
- C3. drift(p, σ, 0) = 0, and drift → 0 as σ → ∞ (the bracket → 0 at every heading): no turn, or a sense that is noise,
  steers nothing.
- C4. |drift| < v = 0.2 mm/s, and strictly, since pirouettes and heading noise are never zero.

**Also predicted beside the drift, not part of the rule**: the pirouette rate λ₀ = 0.01033 per step; the share of
decisions that are decided (outside the dead zone) and, among them, the share whose sign agrees with the noise-free
reading - the sandbox's own "lateral sense right" was 0.986 in the decision audit of 2026-11-04 (a different world,
Hardland, so a pointer, not a comparison).

In [1]:
import platform
from math import erf, sqrt
import numpy as np
print("python", platform.python_version(), "numpy", np.__version__)

V, SH, L, PMAX = 0.2, 0.05, 10, 0.3          # mm/step, heading noise sd, decision interval, pirouette ceiling
GT, ARM, A = 5.0 / 25.0, 1.0, 0.01            # thermal gradient (deg C/mm), lateral arm (mm), dead-zone threshold


def lam0(tau=0.3):
    """Pirouette probability per step with no drive: the turning rule's sigmoid at dD = 0."""
    return PMAX / (1 + np.exp(1 / tau))


def phi(z):
    return 0.5 * (1 + np.vectorize(erf)(z / sqrt(2)))


def steer_probs(th, p, sigma, a=A):
    """P(steer = +1), P(steer = -1), P(steer = 0) at heading th, from the map's share p and the lateral noise sigma.
    The noise-free lateral reading is m = -2 GT ARM sin th (left minus right temperature)."""
    m = -2 * GT * ARM * np.sin(th)
    if sigma == 0:                             # the perfect-sense alternative: no noise, no dead zone
        m = np.where(np.abs(m) < 1e-12, 0.0, m)   # sin(pi) is 1.2e-16, not 0: keep the two poles alike
        pl = (m > 0).astype(float); pr = (m < 0).astype(float)
    else:
        pl = phi((m - a) / sigma)              # left warmer, by the thresholded noisy reading
        pr = phi((-m - a) / sigma)             # right warmer
    plus = p * pl + (1 - p) * pr               # left turn: wants warm and left is warmer, or wants cold and right is
    minus = p * pr + (1 - p) * pl
    return plus, minus, 1 - pl - pr


def torque(th, p, sigma, k):
    """The steering term: expected rotation per step, k (2p - 1) [Phi((m - a)/s) - Phi((-m - a)/s)]."""
    plus, minus, _ = steer_probs(th, p, sigma)
    return k * (plus - minus)


def drift(p, sigma, k, tau=0.3, nb=720, max_cycles=20000, tol=1e-13):
    """Stationary drift (mm/step) up the gradient; also the pirouette rate, the decided share and, among decided
    decisions, the share whose sign agrees with the noise-free reading."""
    th = 2 * np.pi * np.arange(nb) / nb
    n = np.fft.rfftfreq(nb, 1.0 / nb)
    kern = np.exp(-(n ** 2) * SH ** 2 / 2)     # heading noise, as a convolution
    shift = {s: np.exp(-1j * n * k * s) for s in (-1, 0, 1)}   # rotation by k s per step
    lam = lam0(tau)
    plus, minus, none = steer_probs(th, p, sigma)
    probs = {1: plus, -1: minus, 0: none}
    g = np.full(nb, 1.0 / nb)
    drift_val = None
    for _ in range(max_cycles):
        f = {s: g * probs[s] for s in (-1, 0, 1)}
        cos_sum = 0.0
        for _step in range(L):
            for s in (-1, 0, 1):
                q = np.fft.irfft(np.fft.rfft(f[s]) * shift[s] * kern, nb)   # rotate, then noise
                f[s] = (1 - lam) * q + lam * f[s].sum() / nb                # the pirouette, which the noise leaves uniform
            cos_sum += sum((f[s] * np.cos(th)).sum() for s in (-1, 0, 1))
        g_new = sum(f.values()); g_new /= g_new.sum()
        new_drift = V * cos_sum / L
        done = drift_val is not None and np.abs(g_new - g).max() < tol
        g, drift_val = g_new, new_drift
        if done: break
    m = -2 * GT * ARM * np.sin(th)
    m = np.where(np.abs(m) < 1e-12, 0.0, m)
    pl, pr = (phi((m - A) / sigma), phi((-m - A) / sigma)) if sigma else ((m > 0) * 1.0, (m < 0) * 1.0)
    decided = (g * (pl + pr)).sum()
    right = (g * np.where(m > 0, pl, np.where(m < 0, pr, 0.0))).sum() / decided
    return drift_val, lam, decided, right

python 3.13.16 numpy 2.5.4


## 3. The consequences, checked on the code

C1 to C4 are checked numerically, and then the checks are shown to bite: a deliberately wrong theory, with the sign of
the share's effect flipped for one steer direction only (a torque that is no longer odd in sin θ), must fail C1 or C2.
A check that cannot fail has shown nothing. The first wrong rule tried here, `minus = 1 - plus - none`, was the
right rule in disguise (the three already sum to one) and passed both checks; the assertion below caught that, and the
rule was replaced by one in which the share scales left turns and its complement right turns, a distribution that
is not odd in sin θ.

In [2]:
import time
t0 = time.time()
d_half = drift(0.5, 0.2, 0.15)[0]
d_p, d_q = drift(0.8, 0.2, 0.1)[0], drift(0.2, 0.2, 0.1)[0]
d_k0 = drift(1.0, 0.02, 0.0)[0]
d_big = drift(1.0, 1e6, 0.15)[0]
d_max = drift(1.0, 0.0, 0.15)[0]
print(f"C1 drift(p=0.5) = {d_half:+.2e}")
print(f"C2 drift(0.8) + drift(0.2) = {d_p + d_q:+.2e}   (drift(0.8) = {d_p:.5f})")
print(f"C3 drift(k=0) = {d_k0:+.2e}, drift(sigma=1e6) = {d_big:+.2e}")
print(f"C4 drift(p=1, perfect sense, k=0.15) = {d_max:.5f} < {V}")
assert abs(d_half) < 1e-9 and abs(d_p + d_q) < 1e-9 and abs(d_k0) < 1e-9 and abs(d_big) < 1e-6 and 0 < d_max < V

# Red-first: the same chain with a wrong steer rule (the map's share applied to left turns only) must fail C1 and C2.
_orig = steer_probs
def _wrong(th, p, sigma, a=A):
    plus, minus, none = _orig(th, p, sigma, a)
    m = -2 * GT * ARM * np.sin(th); pl = phi((m - a) / sigma); pr = phi((-m - a) / sigma)
    return p * pl, (1 - p) * pr, 1 - p * pl - (1 - p) * pr   # the share scales left turns only: not odd in sin theta
steer_probs = _wrong
w_half = drift(0.5, 0.2, 0.15)[0]; w_sum = drift(0.8, 0.2, 0.1)[0] + drift(0.2, 0.2, 0.1)[0]
steer_probs = _orig
print(f"red-first: wrong rule gives drift(0.5) = {w_half:+.4f}, drift(0.8) + drift(0.2) = {w_sum:+.4f}")
assert abs(w_half) > 1e-3 or abs(w_sum) > 1e-3
print(f"{time.time() - t0:.1f} s")

C1 drift(p=0.5) = -7.36e-18
C2 drift(0.8) + drift(0.2) = -4.16e-17   (drift(0.8) = 0.09485)
C3 drift(k=0) = -1.31e-17, drift(sigma=1e6) = +3.75e-08
C4 drift(p=1, perfect sense, k=0.15) = 0.14646 < 0.2
red-first: wrong rule gives drift(0.5) = +0.1024, drift(0.8) + drift(0.2) = +0.1820
0.5 s


## 4. The predictions, before any simulation

The grid: share p ∈ {0.2, 0.5, 0.65, 0.8, 1.0} - 0.2 and 0.8 for C2, 0.5 for C1, 0.65 near the decision audit's
mean of 0.513 and its upper bound 0.619, 1.0 the ceiling - lateral noise σ ∈ {0.02, 0.2, 0.6} - the sandbox's own,
then ten and thirty times it, against a signal of amplitude 0.4 - and turn size k ∈ {0.05, 0.1, 0.15} rad per step -
a third, two thirds and the sandbox's own. 45 conditions. Beside the full theory, the *perfect-sense* alternative
(σ → 0, no dead zone), which says the noise does not matter, plays the part the instantaneous theory played on
2026-10-26: the full theory must beat it, or the noise link has not been shown to be understood.

In [3]:
SHARES, SIGMAS, KS = (0.2, 0.5, 0.8, 1.0, 0.65), (0.02, 0.2, 0.6), (0.05, 0.1, 0.15)
GRID = [(p, s, k) for p in SHARES for s in SIGMAS for k in KS]      # the order steer_sim.py indexes
t0 = time.time()
PRED = {}
for i, (p, s, k) in enumerate(GRID):
    d, lam, dec, right = drift(p, s, k)
    alt = drift(p, 0.0, k)[0]
    PRED[i] = (d, alt, lam, dec, right)
print(f"{len(GRID)} conditions in {time.time() - t0:.1f} s; lambda0 = {lam:.5f} per step\n")
print("Predicted drift, mm/s (full theory | perfect-sense alternative), columns sigma x k:")
hdr = "  ".join(f"s{s:<4} k{k:<4}" for s in SIGMAS for k in KS)
print(f"p     {hdr}")
for p in sorted(SHARES):
    row = [PRED[GRID.index((p, s, k))] for s in SIGMAS for k in KS]
    print(f"{p:<5} " + "  ".join(f"{d:+.4f}|{a:+.4f}" for d, a, *_ in row))
print("\nPredicted decided share and lateral-sign agreement (full theory), by sigma, at p = 1, k = 0.15:")
for s in SIGMAS:
    _, _, _, dec, right = PRED[GRID.index((1.0, s, 0.15))]
    print(f"  sigma {s}: decided {dec:.4f}, sign right {right:.4f}")

45 conditions in 6.8 s; lambda0 = 0.01033 per step

Predicted drift, mm/s (full theory | perfect-sense alternative), columns sigma x k:
p     s0.02 k0.05  s0.02 k0.1   s0.02 k0.15  s0.2  k0.05  s0.2  k0.1   s0.2  k0.15  s0.6  k0.05  s0.6  k0.1   s0.6  k0.15
0.2   -0.1191|-0.1188  -0.1102|-0.1097  -0.0898|-0.0893  -0.1049|-0.1188  -0.0949|-0.1097  -0.0762|-0.0893  -0.0572|-0.1188  -0.0472|-0.1097  -0.0358|-0.0893
0.5   -0.0000|-0.0000  -0.0000|-0.0000  -0.0000|-0.0000  -0.0000|-0.0000  -0.0000|-0.0000  -0.0000|-0.0000  -0.0000|-0.0000  -0.0000|-0.0000  -0.0000|-0.0000
0.65  +0.0710|+0.0707  +0.0592|+0.0589  +0.0452|+0.0449  +0.0609|+0.0707  +0.0504|+0.0589  +0.0383|+0.0449  +0.0299|+0.0707  +0.0239|+0.0589  +0.0179|+0.0449
0.8   +0.1191|+0.1188  +0.1102|+0.1097  +0.0898|+0.0893  +0.1049|+0.1188  +0.0949|+0.1097  +0.0762|+0.0893  +0.0572|+0.1188  +0.0472|+0.1097  +0.0358|+0.0893
1.0   +0.1526|+0.1526  +0.1580|+0.1577  +0.1471|+0.1465  +0.1381|+0.1526  +0.1389|+0.1577  +0.1254|+0.1465  +0

## 5. Pre-registration

Written into the journal of 2026-11-08 before the runs, and repeated here so that the notebook is self-contained.

- **The simulation**: gr-fec's `EvoBrain` (`evolve.py`) with every drive, reward, trust, roaming and curiosity gene
  0, `eta` 10⁻⁵, `s0` 10⁻², `tau` 0.3, `integ` 10, unedited, in a body that is `DirBody` (`steer.py`) with its lateral
  noise and its turn made parameters (`steer_sim.py`, `SteerBody`: the same two formulas with 0.02 and 0.15 replaced by
  σ and k), on `World(thermal=True)` with the wall removed (radius 10¹² mm). The evaluation-side wrapper, at every tenth
  step, draws "wants warmer" with probability p from a generator of its own (seed + 10⁶), reads the body's noisy
  lateral sense as the greedy rule does (threshold 0.01) and sets the body's steer by the greedy rule's four cases.
  The wrapper is an evaluation tool for a toy world; it reads ground truth (which way is home) only to impose the
  share, and nothing of this is a brain.
- **Per condition**: 2000 worms from the origin, uniform headings, 4000 steps of one second; the drift is each worm's
  x-displacement from step 500 to 4000 divided by 3500 s, averaged over worms, with its standard error over worms.
  Seeds 54000 + i for condition i in the grid's order. Two processes. `python3 -I <s>/steer8e/steer_sim.py <i>`
  from the repository's root, Python 3.13.16, numpy 2.5.4.
- **Rules, fixed beforehand.** A condition *agrees* when |theory − simulation| ≤ max(0.15 × |theory|, 3 × SE). The
  allowance is 3 standard errors rather than the 2 of 2026-10-26 because nine conditions predict exactly zero, where a
  relative tolerance is empty and a 2-SE band would fail about one condition in twenty by chance alone.
    - **T1**: the full theory agrees in at least 41 of 45 conditions (the 2026-10-26 bar, 32 of 36, scaled).
    - **T1b**: over the 36 conditions with a non-zero prediction, the full theory's median |relative error| is at most
      0.10 - the 2026-10-26 theory reached 0.031 - and below the perfect-sense alternative's.
    - **T2**, the symmetry: for each of the nine (σ, k) pairs, simulation(0.2) + simulation(0.8) lies within
      3 √(SE₀.₂² + SE₀.₈²) of zero, in at least 8 of 9.
- **What refutes the term.** T1 failing refutes the steering term as stated. Failures patterned by σ (at 0.6 and not
  at 0.02) blame the sense link, the Gaussian-threshold account of the dead zone; failures patterned by k and p (at
  0.15 and 1.0) blame the within-cycle chain, the held turn against pirouettes; T1b's second clause failing means the
  noise link adds nothing shown. T2 failing refutes the reading that the sandbox's body and plate are
  mirror-symmetric about the gradient, independently of the theory.
- **What was seen before this was written**, disclosed: one smoke run of `steer_sim.py` on condition 44 (p 0.65,
  σ 0.6, k 0.15) with seed 1, 400 steps and a burn-in of 100, to see that the wrapper runs and to time it: it printed a
  drift of 0.01812 (SE 0.00073), a pirouette rate of 0.0103, a decided share of 0.9878 and a lateral-sign agreement
  of 0.6637. It set nothing above; the theory's predictions for that condition were computed by this notebook after
  the rules were written.

## 6. Results, added after the run

The run was `nohup bash <s>/steer8e/run.sh &` from 23:17:30 UTC, two seconds after the pre-registration was pushed in `ce963ea`, to 23:36:02 UTC (the `done` marker), 45 of 45 conditions, Python 3.13.16, numpy 2.5.4. The 45 printed lines are pasted below as the notebook's data (their SHA-256 and the
scripts are in the pull request), so that this notebook runs from a fresh kernel with numpy alone.

In [4]:
RESULTS_TEXT = """\
cond 0 seed 54000 p 0.2 sigma 0.02 k 0.05: sim -0.11964 (SE 0.00032) pir 0.0103 decided 0.9694 lat right 0.9874
cond 1 seed 54001 p 0.2 sigma 0.02 k 0.1: sim -0.11015 (SE 0.00023) pir 0.0103 decided 0.9794 lat right 0.9917
cond 2 seed 54002 p 0.2 sigma 0.02 k 0.15: sim -0.08987 (SE 0.00019) pir 0.0103 decided 0.9828 lat right 0.9929
cond 3 seed 54003 p 0.2 sigma 0.2 k 0.05: sim -0.10461 (SE 0.00034) pir 0.0103 decided 0.9777 lat right 0.8247
cond 4 seed 54004 p 0.2 sigma 0.2 k 0.1: sim -0.09499 (SE 0.00023) pir 0.0104 decided 0.9796 lat right 0.8487
cond 5 seed 54005 p 0.2 sigma 0.2 k 0.15: sim -0.07630 (SE 0.00020) pir 0.0103 decided 0.9813 lat right 0.8619
cond 6 seed 54006 p 0.2 sigma 0.6 k 0.05: sim -0.05807 (SE 0.00043) pir 0.0103 decided 0.9880 lat right 0.6576
cond 7 seed 54007 p 0.2 sigma 0.6 k 0.1: sim -0.04711 (SE 0.00030) pir 0.0104 decided 0.9882 lat right 0.6608
cond 8 seed 54008 p 0.2 sigma 0.6 k 0.15: sim -0.03605 (SE 0.00021) pir 0.0103 decided 0.9881 lat right 0.6641
cond 9 seed 54009 p 0.5 sigma 0.02 k 0.05: sim 0.00067 (SE 0.00049) pir 0.0103 decided 0.9824 lat right 0.9929
cond 10 seed 54010 p 0.5 sigma 0.02 k 0.1: sim -0.00058 (SE 0.00032) pir 0.0104 decided 0.9825 lat right 0.9929
cond 11 seed 54011 p 0.5 sigma 0.02 k 0.15: sim -0.00017 (SE 0.00022) pir 0.0103 decided 0.9830 lat right 0.9930
cond 12 seed 54012 p 0.5 sigma 0.2 k 0.05: sim -0.00057 (SE 0.00050) pir 0.0103 decided 0.9814 lat right 0.8638
cond 13 seed 54013 p 0.5 sigma 0.2 k 0.1: sim 0.00010 (SE 0.00032) pir 0.0104 decided 0.9810 lat right 0.8638
cond 14 seed 54014 p 0.5 sigma 0.2 k 0.15: sim 0.00035 (SE 0.00022) pir 0.0104 decided 0.9811 lat right 0.8627
cond 15 seed 54015 p 0.5 sigma 0.6 k 0.05: sim 0.00036 (SE 0.00049) pir 0.0103 decided 0.9881 lat right 0.6633
cond 16 seed 54016 p 0.5 sigma 0.6 k 0.1: sim -0.00000 (SE 0.00031) pir 0.0104 decided 0.9882 lat right 0.6631
cond 17 seed 54017 p 0.5 sigma 0.6 k 0.15: sim 0.00042 (SE 0.00021) pir 0.0104 decided 0.9879 lat right 0.6639
cond 18 seed 54018 p 0.8 sigma 0.02 k 0.05: sim 0.11906 (SE 0.00031) pir 0.0103 decided 0.9697 lat right 0.9877
cond 19 seed 54019 p 0.8 sigma 0.02 k 0.1: sim 0.11044 (SE 0.00023) pir 0.0103 decided 0.9797 lat right 0.9917
cond 20 seed 54020 p 0.8 sigma 0.02 k 0.15: sim 0.08973 (SE 0.00019) pir 0.0103 decided 0.9826 lat right 0.9931
cond 21 seed 54021 p 0.8 sigma 0.2 k 0.05: sim 0.10389 (SE 0.00033) pir 0.0104 decided 0.9778 lat right 0.8255
cond 22 seed 54022 p 0.8 sigma 0.2 k 0.1: sim 0.09492 (SE 0.00024) pir 0.0103 decided 0.9798 lat right 0.8487
cond 23 seed 54023 p 0.8 sigma 0.2 k 0.15: sim 0.07576 (SE 0.00019) pir 0.0103 decided 0.9812 lat right 0.8614
cond 24 seed 54024 p 0.8 sigma 0.6 k 0.05: sim 0.05682 (SE 0.00045) pir 0.0103 decided 0.9878 lat right 0.6568
cond 25 seed 54025 p 0.8 sigma 0.6 k 0.1: sim 0.04695 (SE 0.00029) pir 0.0103 decided 0.9882 lat right 0.6621
cond 26 seed 54026 p 0.8 sigma 0.6 k 0.15: sim 0.03584 (SE 0.00021) pir 0.0104 decided 0.9883 lat right 0.6625
cond 27 seed 54027 p 1.0 sigma 0.02 k 0.05: sim 0.15255 (SE 0.00021) pir 0.0103 decided 0.9583 lat right 0.9826
cond 28 seed 54028 p 1.0 sigma 0.02 k 0.1: sim 0.15791 (SE 0.00013) pir 0.0104 decided 0.9751 lat right 0.9899
cond 29 seed 54029 p 1.0 sigma 0.02 k 0.15: sim 0.14704 (SE 0.00010) pir 0.0103 decided 0.9820 lat right 0.9927
cond 30 seed 54030 p 1.0 sigma 0.2 k 0.05: sim 0.13793 (SE 0.00023) pir 0.0103 decided 0.9746 lat right 0.7887
cond 31 seed 54031 p 1.0 sigma 0.2 k 0.1: sim 0.13882 (SE 0.00014) pir 0.0103 decided 0.9777 lat right 0.8279
cond 32 seed 54032 p 1.0 sigma 0.2 k 0.15: sim 0.12515 (SE 0.00013) pir 0.0104 decided 0.9808 lat right 0.8590
cond 33 seed 54033 p 1.0 sigma 0.6 k 0.05: sim 0.08773 (SE 0.00038) pir 0.0103 decided 0.9878 lat right 0.6490
cond 34 seed 54034 p 1.0 sigma 0.6 k 0.1: sim 0.07591 (SE 0.00026) pir 0.0103 decided 0.9879 lat right 0.6576
cond 35 seed 54035 p 1.0 sigma 0.6 k 0.15: sim 0.05953 (SE 0.00020) pir 0.0104 decided 0.9879 lat right 0.6631
cond 36 seed 54036 p 0.65 sigma 0.02 k 0.05: sim 0.07045 (SE 0.00043) pir 0.0103 decided 0.9783 lat right 0.9913
cond 37 seed 54037 p 0.65 sigma 0.02 k 0.1: sim 0.05914 (SE 0.00029) pir 0.0103 decided 0.9818 lat right 0.9926
cond 38 seed 54038 p 0.65 sigma 0.02 k 0.15: sim 0.04469 (SE 0.00022) pir 0.0104 decided 0.9829 lat right 0.9933
cond 39 seed 54039 p 0.65 sigma 0.2 k 0.05: sim 0.06149 (SE 0.00044) pir 0.0103 decided 0.9802 lat right 0.8515
cond 40 seed 54040 p 0.65 sigma 0.2 k 0.1: sim 0.04992 (SE 0.00029) pir 0.0103 decided 0.9809 lat right 0.8591
cond 41 seed 54041 p 0.65 sigma 0.2 k 0.15: sim 0.03850 (SE 0.00021) pir 0.0104 decided 0.9813 lat right 0.8620
cond 42 seed 54042 p 0.65 sigma 0.6 k 0.05: sim 0.02973 (SE 0.00046) pir 0.0104 decided 0.9883 lat right 0.6614
cond 43 seed 54043 p 0.65 sigma 0.6 k 0.1: sim 0.02354 (SE 0.00030) pir 0.0104 decided 0.9882 lat right 0.6633
cond 44 seed 54044 p 0.65 sigma 0.6 k 0.15: sim 0.01761 (SE 0.00022) pir 0.0104 decided 0.9880 lat right 0.6637
"""
import re
SIM = {}
for line in RESULTS_TEXT.strip().splitlines():
    m = re.match(r"cond (\d+) seed (\d+) p ([\d.]+) sigma ([\d.]+) k ([\d.]+): sim (\S+) \(SE (\S+)\) pir (\S+) decided (\S+) lat right (\S+)", line)
    i = int(m.group(1)); SIM[i] = tuple(float(m.group(j)) for j in range(6, 11))
    assert GRID[i] == tuple(float(m.group(j)) for j in (3, 4, 5)) and int(m.group(2)) == 54000 + i
print(len(SIM), "conditions parsed")

agree = {}; rel_full = []; rel_alt = []
print(f"{'i':>2} {'p':>4} {'sigma':>5} {'k':>4} | {'sim':>8} {'SE':>7} | {'theory':>8} {'rel':>7} | {'alt':>8} {'rel':>7} | agree")
for i, (p, s, k) in enumerate(GRID):
    d, alt, lam, dec, right = PRED[i]; sim, se, pir, sdec, sright = SIM[i]
    ok = abs(d - sim) <= max(0.15 * abs(d), 3 * se); agree[i] = ok
    if abs(d) > 1e-9:
        rel_full.append(abs(d - sim) / abs(sim)); rel_alt.append(abs(alt - sim) / abs(sim))
        rf, ra = f"{(d - sim) / sim:+.3f}", f"{(alt - sim) / sim:+.3f}"
    else:
        rf = ra = f"{(sim) / se:+.1f}SE"
    print(f"{i:>2} {p:>4} {s:>5} {k:>4} | {sim:+.5f} {se:.5f} | {d:+.5f} {rf:>7} | {alt:+.5f} {ra:>7} | {'yes' if ok else 'NO'}")
n_agree = sum(agree.values()); med_full = float(np.median(rel_full)); med_alt = float(np.median(rel_alt))
# the perfect-sense alternative under the same agreement rule, for the record
n_alt = sum(abs(PRED[i][1] - SIM[i][0]) <= max(0.15 * abs(PRED[i][1]), 3 * SIM[i][1]) for i in range(45))
print(f"\nT1: full theory agrees in {n_agree} of 45 (bar 41): {'holds' if n_agree >= 41 else 'FAILS'};"
      f" the perfect-sense alternative in {n_alt} of 45")
print(f"T1b: median |relative error| over {len(rel_full)} non-zero conditions: full {med_full:.4f}, alternative {med_alt:.4f}"
      f" (bar: full <= 0.10 and below the alternative): {'holds' if med_full <= 0.10 and med_full < med_alt else 'FAILS'}")
t2 = 0
for s in SIGMAS:
    for k in KS:
        a, b = SIM[GRID.index((0.2, s, k))], SIM[GRID.index((0.8, s, k))]
        tot = a[0] + b[0]; band = 3 * np.hypot(a[1], b[1]); ok = abs(tot) <= band; t2 += ok
        print(f"T2 sigma {s} k {k}: sim(0.2) + sim(0.8) = {tot:+.5f}, band {band:.5f}: {'within' if ok else 'OUTSIDE'}")
print(f"T2: {t2} of 9 within (bar 8): {'holds' if t2 >= 8 else 'FAILS'}")
print("\nBeside the rule: pirouette rate, decided share and lateral-sign agreement, simulation against theory")
for s in SIGMAS:
    i = GRID.index((1.0, s, 0.15)); sim, se, pir, sdec, sright = SIM[i]; d, alt, lam, dec, right = PRED[i]
    print(f"  p 1 sigma {s} k 0.15: pir {pir:.4f} vs {lam:.4f}; decided {sdec:.4f} vs {dec:.4f}; sign right {sright:.4f} vs {right:.4f}")
signed = [(PRED[i][0] - SIM[i][0]) / SIM[i][0] for i in range(45) if abs(PRED[i][0]) > 1e-9]
print(f"\nsigned relative error, full theory: median {np.median(signed):+.4f}, above the simulation in {sum(x > 0 for x in signed)} of {len(signed)}")

45 conditions parsed
 i    p sigma    k |      sim      SE |   theory     rel |      alt     rel | agree
 0  0.2  0.02 0.05 | -0.11964 0.00032 | -0.11906  -0.005 | -0.11885  -0.007 | yes
 1  0.2  0.02  0.1 | -0.11015 0.00023 | -0.11019  +0.000 | -0.10966  -0.004 | yes
 2  0.2  0.02 0.15 | -0.08987 0.00019 | -0.08979  -0.001 | -0.08927  -0.007 | yes
 3  0.2   0.2 0.05 | -0.10461 0.00034 | -0.10488  +0.003 | -0.11885  +0.136 | yes
 4  0.2   0.2  0.1 | -0.09499 0.00023 | -0.09485  -0.001 | -0.10966  +0.154 | yes
 5  0.2   0.2 0.15 | -0.07630 0.00020 | -0.07619  -0.001 | -0.08927  +0.170 | yes
 6  0.2   0.6 0.05 | -0.05807 0.00043 | -0.05725  -0.014 | -0.11885  +1.047 | yes
 7  0.2   0.6  0.1 | -0.04711 0.00030 | -0.04718  +0.001 | -0.10966  +1.328 | yes
 8  0.2   0.6 0.15 | -0.03605 0.00021 | -0.03580  -0.007 | -0.08927  +1.476 | yes
 9  0.5  0.02 0.05 | +0.00067 0.00049 | -0.00000  +1.4SE | -0.00000  +1.4SE | yes
10  0.5  0.02  0.1 | -0.00058 0.00032 | -0.00000  -1.8SE | -0.00000  -1.8SE

In [5]:
import matplotlib
matplotlib.use("svg")
import matplotlib.pyplot as plt
plt.rcParams["svg.hashsalt"] = "steering-term-8e"
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
colours = {0.05: "#1f6feb", 0.1: "#d29922", 0.15: "#8b949e"}
ps = sorted(SHARES)
for ax, s in zip(axes, SIGMAS):
    for k in KS:
        idx = [GRID.index((p, s, k)) for p in ps]
        ax.plot(ps, [PRED[i][0] for i in idx], "-", color=colours[k], lw=1.2, label=f"theory, k {k}")
        ax.plot(ps, [PRED[i][1] for i in idx], ":", color=colours[k], lw=0.9)
        ax.errorbar(ps, [SIM[i][0] for i in idx], yerr=[3 * SIM[i][1] for i in idx], fmt="o", ms=3.5, color=colours[k], capsize=2, lw=0.9)
    ax.axhline(0, color="#999", lw=0.5); ax.set_title(f"lateral noise sigma {s}", fontsize=9); ax.set_xlabel("share pointing home, p")
axes[0].set_ylabel("drift up the gradient, mm/s"); axes[0].legend(frameon=False, fontsize=7, loc="upper left")
fig.text(0.5, -0.03, "lines: the full theory; dotted: the perfect-sense alternative; points: simulation with 3 SE bars", ha="center", fontsize=8)
fig.tight_layout()
fig.savefig("../docs/figures/steering_term_8e.svg", metadata={"Date": None}, bbox_inches="tight")
plt.close(fig); print("written docs/figures/steering_term_8e.svg")

written docs/figures/steering_term_8e.svg


## 7. Reading

- **T1 holds**: the full theory agrees in 45 of 45 conditions (bar 41); the perfect-sense alternative in 31, failing
  in all twelve conditions at thirty times the sandbox's noise and in two of twelve at ten times.
- **T1b holds**: the full theory's median |relative error| over the 36 non-zero conditions is 0.0022, an order of
  magnitude under the 2026-10-26 turning theory's 0.031 and under the bar of 0.10; the alternative's is 0.155. The
  largest error is 1.8% (p 0.65, σ 0.6, k 0.15), and the signed errors are not biased to one side (median +0.0006,
  above the simulation in 22 of 36), unlike the turning theory, which ran high by about 3%. That is consistent with A5:
  the chain is exact to rounding where the earlier theory averaged a correlated noise.
- **T2 holds**: all nine pairs sum to zero within their bands, so the sandbox's body and plate are mirror-symmetric
  about the gradient to the resolution of 2000 worms, as the reading of the code said.
- Beside the rule, the pirouette rate, the decided share and the lateral-sign agreement match the theory to three
  decimals at every noise level.
- The two things the table said before the run are seen in the simulation too: below p 1 the smallest turn drifts
  fastest, at p 1 the middle one does.
- **What it does not show.** The term is tested with an imposed, heading-independent share (A1), zero drive (A4), no
  wall and a uniform gradient (A3), and the sandbox's body noise. A learned map's share depends on where the worm is,
  Hardland has walls and a band, and gr-fec's worms are driven. Step (f) imposes a fixed *map* instead of a coin,
  and composing the steering term with the driven turning rule is the gap after that.
- Still a toy world.